# Mozar modeling

M0 detector score, M1 history model, M2 both combined. Same posts, same splits.
Posts by one author, and posts of one picture, always stay on the same side of a split.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from config import DATA, RESULTS
RESULTS.mkdir(exist_ok=True)

feat = pd.read_csv(DATA / "features.csv", dtype={"pair_id": str})
feat["y"] = (feat.label == "manipulated").astype(int)
feat.label.value_counts()

## 1. Data

In [ ]:
print(pd.crosstab(feat.subreddit, feat.label))
print(feat.groupby("label")[["n_posts", "n_comments", "n_subreddits", "prior_repost_share", "has_history"]].mean().round(2))

In [ ]:
from IPython.display import Image, display
dec = pd.read_csv(DATA / "labels/decisions.csv") if (DATA / "labels/decisions.csv").exists() else pd.DataFrame()
for pid in dec.pair_id.head(3) if len(dec) else []:
    display(Image(str(DATA / f"pairs/outline/{pid}_a.jpg"), width=350), Image(str(DATA / f"pairs/outline/{pid}_b.jpg"), width=350))

## 2. Splits

Groups join every post that shares an author or a picture, so neither can sit on both sides.

In [ ]:
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, p, img in zip(feat.author, feat.pair_id.fillna(""), feat.img):
    parent[find("a:" + a)] = find("i:" + (p or img))
feat["group"] = [find("a:" + a) for a in feat.author]

def cv_scores(X, y, groups, make_model, seed=0, folds=5):
    """Out-of-fold scores from grouped, stratified cross-validation."""
    out = np.full(len(y), np.nan)
    k = min(folds, y.sum(), (1 - y).sum())
    for tr, te in StratifiedGroupKFold(k, shuffle=True, random_state=seed).split(X, y, groups):
        out[te] = make_model().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return out

def auc_ci(y, s, n=1000, seed=0):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): vals.append(roc_auc_score(y[i], s[i]))
    return roc_auc_score(y, s), np.percentile(vals, [2.5, 97.5])

## 3. M0: detector score

Filled in once FLAME / TruFor scores exist in `results/m0_scores.csv` (columns: img, score).

In [ ]:
m0_file = RESULTS / "m0_scores.csv"
if m0_file.exists():
    feat = feat.merge(pd.read_csv(m0_file), on="img", how="left").rename(columns={"score": "m0_raw"})
    feat["m0"] = cv_scores(feat[["m0_raw"]].values, y, g, lambda: LogisticRegression())  # calibrated on Reddit images
    print("M0 AUC %.3f (%.3f to %.3f)" % ((lambda a, c: (a, *c))(*auc_ci(y, feat.m0.values))))
else:
    print("No detector scores yet.")

## 4. M1: history model (logistic regression)

In [ ]:
HIST = ["n_posts", "n_comments", "n_subreddits", "subreddit_entropy", "share_image_posts",
        "days_observed", "posts_per_day", "n_prior_pool_posts", "prior_repost_share", "prior_variant_share"]
X, y, g = np.log1p(feat[HIST].values.astype(float)), feat.y.values, feat.group.values
logreg = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
if y.sum() >= 2:
    feat["m1"] = cv_scores(X, y, g, logreg)
    auc, ci = auc_ci(y, feat.m1.values)
    print(f"M1 AUC {auc:.3f}  (95% interval {ci[0]:.3f} to {ci[1]:.3f})  on {y.sum()} manipulated / {len(y)} posts")
else:
    print(f"Only {y.sum()} manipulated posts: too few to fit M1 yet.")

## 5. M2: both combined (score plus score)

In [ ]:
if "m0" in feat and "m1" in feat:
    logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    Z = np.c_[logit(feat.m0), logit(feat.m1)]
    feat["m2"] = cv_scores(Z, y, g, lambda: LogisticRegression())
    print("M2 AUC %.3f (%.3f to %.3f)" % ((lambda a, c: (a, *c))(*auc_ci(y, feat.m2.values))))

## 6. Checks

Shuffled history must fall to chance. More checks (planted signal, repost-only baseline) go here.

In [ ]:
if "m1" in feat:
    rng = np.random.default_rng(1)
    Xs = X.copy()
    for s in feat.subreddit.unique():  # shuffle within community
        idx = np.where(feat.subreddit.values == s)[0]
        Xs[idx] = Xs[rng.permutation(idx)]
    print("shuffled-history AUC %.3f" % roc_auc_score(y, cv_scores(Xs, y, g, logreg)))

## 7. Save scores

In [ ]:
feat.to_csv(RESULTS / "scores.csv", index=False)